In [31]:
import os
import re
import time
import pandas as pd
import geemap, ee
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns


# --- 可配置参数 ---
CSV_PATH = r"E:\NWP\CS2_S1_matched\time_match_2023.csv"   # 本地CSV，需包含sceneName列
SCENE_COL = "sceneName"                                    # S1产品名列名
S2_CLOUD_MAX = 20                                          # S2 影像层面云量（%）上限
L8_CLOUD_MAX = 20                                          # L8/9 影像层面云量（%）上限
TIME_WINDOW_HOURS = 24                                      # 时间窗口：±1小时
LIMIT_PER_SENSOR = 1                                       # 每个S1匹配各传感器保留多少条（按云量最小）
DO_DRIVE_EXPORT = False                                    # 是否导出Drive（默认False，先确认匹配结果）
S2_EXPORT_SCALE = 20                                       # Sentinel-2 导出分辨率
L8_EXPORT_SCALE = 30                                       # Landsat 导出分辨率

# 输出文件路径
OUTPUT_DIR = r"E:\NWP\CS2_S1_matched\S1_S2_overlap"                             # 输出目录
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    
# --- 初始化EE账户（会弹浏览器登录一次） ---
geemap.ee_initialize()


In [32]:

# 数据集（GEE）
S1_IC = ee.ImageCollection('COPERNICUS/S1_GRD')  # S1
S2_IC = ee.ImageCollection('COPERNICUS/S2')
L8_IC = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
         .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')))  # Landsat-8/9 L2
# 工具函数
def is_ew_mode(scene_name):
    """检查是否为EW模式的S1数据"""
    return '_EW_' in scene_name.upper()

def parse_s1_time_from_name(name):
    """从 S1 名称中提取起止时间（UTC），常见命名：S1A_EW_GRDM_1SDH_YYYYMMDDTHHMMSS_YYYYMMDDTHHMMSS_..."""
    m = re.search(r'_(\d{8}T\d{6})_(\d{8}T\d{6})_', name)
    if not m:
        return None, None
    def to_ms(s):
        # 转毫秒
        # 例如 20230630T134853
        from datetime import datetime, timezone
        dt = datetime.strptime(s, "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)
        return int(dt.timestamp() * 1000)
    return to_ms(m.group(1)), to_ms(m.group(2))

def get_s1_image_by_index(name):
    """以 system:index 精确匹配"""
    col = S1_IC.filter(ee.Filter.eq('system:index', name))
    img = ee.Image(col.first())
    return img

def ms_to_iso(ms):
    return ee.Date(ms).format('YYYY-MM-dd HH:mm:ss').getInfo()

def find_overlaps_for_s1(s1_name):
    record_list = []
    s1_img = get_s1_image_by_index(s1_name)
    # 若GEE中找不到，尝试从名字解析时间，后续仅时间过滤（无空间几何则跳过）
    try:
        s1_geom = s1_img.geometry()
        # 触发一次取属性，若为空会报错
        _ = s1_geom.area(1).getInfo()
        has_geom = True
    except Exception:
        has_geom = False

    # 时间窗口
    if has_geom:
        t_start = ee.Number(s1_img.get('system:time_start')).getInfo()
        # S1 有时有停止时间，可用中点，但用 start ±1h 已够严格
    else:
        # 仅从名称解析
        t0, t1 = parse_s1_time_from_name(s1_name)
        if t0 is None:
            print(f"[WARN] 无法在GEE找到S1影像，且无法从名称解析时间：{s1_name}")
            return []
        # 用起始时间作为窗口中心
        t_start = t0

    ms_hour = 60 * 60 * 1000
    t_min = t_start - TIME_WINDOW_HOURS * ms_hour
    t_max = t_start + TIME_WINDOW_HOURS * ms_hour

    # 构建过滤（时间 + 空间 + 质量）
    def filter_and_collect(ic, cloud_prop, cloud_max, sensor_tag):
        ic2 = ic.filterDate(ee.Date(t_min), ee.Date(t_max))
        if has_geom:
            ic2 = ic2.filterBounds(s1_geom)
        ic2 = ic2.filter(ee.Filter.lte(cloud_prop, cloud_max))

        # 按云量升序
        ic2 = ic2.sort(cloud_prop)

        # 收集前 N 个结果
        imgs = ic2.toList(LIMIT_PER_SENSOR)
        n = imgs.size().getInfo()
        for i in range(n):
            img = ee.Image(imgs.get(i))
            img_id = img.get('system:index').getInfo()
            t_img = img.get('system:time_start').getInfo()
            cloud = img.get(cloud_prop).getInfo() if img.get(cloud_prop) else None

            inter_km2 = None
            if has_geom:
                try:
                    inter = s1_geom.intersection(img.geometry(), ee.ErrorMargin(1))
                    inter_km2 = inter.area(1).divide(1e6).getInfo()
                except Exception:
                    inter_km2 = None

            record_list.append({
                "s1_scene": s1_name,
                "s1_time_utc": ms_to_iso(t_start),
                "sensor": sensor_tag,
                "match_id": img_id,
                "match_time_utc": ms_to_iso(t_img),
                "cloud_%": cloud,
                "intersect_km2": inter_km2
            })

            # 可选导出：裁剪到 S1 footprint
            if DO_DRIVE_EXPORT and has_geom:
                out_name = f"{sensor_tag}_{img_id}_clip_{s1_name[:24]}"
                scale = S2_EXPORT_SCALE if sensor_tag == "S2" else L8_EXPORT_SCALE
                try:
                    geemap.ee_export_image_to_drive(
                        image=img.clip(s1_geom),
                        description=out_name[:95],
                        scale=scale,
                        region=s1_geom,
                        maxPixels=1e13,
                        fileFormat='GeoTIFF'
                    )
                    print(f"[EXPORT] 已提交导出任务：{out_name}")
                except Exception as e:
                    print(f"[EXPORT-ERR] {out_name}: {e}")

    # S2
    filter_and_collect(S2_IC, 'CLOUDY_PIXEL_PERCENTAGE', S2_CLOUD_MAX, 'S2')
    # L8/9
    filter_and_collect(L8_IC, 'CLOUD_COVER', L8_CLOUD_MAX, 'L8_9')

    return record_list

def create_s1_s2_match_csv(res_df):
    """创建S1-S2匹配文件名的CSV"""
    s1_s2_matches = []
    
    # 筛选出S2的匹配结果
    s2_matches = res_df[res_df['sensor'] == 'S2'].copy()
    
    for _, row in s2_matches.iterrows():
        s1_s2_matches.append({
            'S1_scene_name': row['s1_scene'],
            'S2_scene_id': row['match_id'],
            'S1_time_utc': row['s1_time_utc'],
            'S2_time_utc': row['match_time_utc'],
            'cloud_percentage': row['cloud_%'],
            'intersection_km2': row['intersect_km2'],
            'time_diff_minutes': abs(pd.to_datetime(row['s1_time_utc']) - 
                                   pd.to_datetime(row['match_time_utc'])).total_seconds() / 60
        })
    
    match_df = pd.DataFrame(s1_s2_matches)
    return match_df

def visualize_results(res_df):
    """创建可视化图表"""
    if len(res_df) == 0:
        print("没有数据可供可视化")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. 云量分布
    axes[0,0].hist(res_df['cloud_%'].dropna(), bins=20, alpha=0.7, color='skyblue')
    axes[0,0].set_title('Cloud Coverage Distribution')
    axes[0,0].set_xlabel('Cloud Percentage (%)')
    axes[0,0].set_ylabel('Frequency')
    
    # 2. 传感器匹配数量
    sensor_counts = res_df['sensor'].value_counts()
    axes[0,1].pie(sensor_counts.values, labels=sensor_counts.index, autopct='%1.1f%%')
    axes[0,1].set_title('Matches by Sensor')
    
    # 3. 交集面积分布
    if 'intersect_km2' in res_df.columns:
        intersect_data = res_df['intersect_km2'].dropna()
        if len(intersect_data) > 0:
            axes[1,0].hist(intersect_data, bins=20, alpha=0.7, color='lightgreen')
            axes[1,0].set_title('Intersection Area Distribution')
            axes[1,0].set_xlabel('Area (km²)')
            axes[1,0].set_ylabel('Frequency')
        else:
            axes[1,0].text(0.5, 0.5, 'No intersection data available', 
                          ha='center', va='center', transform=axes[1,0].transAxes)
    
    # 4. 云量 vs 交集面积散点图
    if 'intersect_km2' in res_df.columns:
        valid_data = res_df.dropna(subset=['cloud_%', 'intersect_km2'])
        if len(valid_data) > 0:
            scatter = axes[1,1].scatter(valid_data['cloud_%'], valid_data['intersect_km2'], 
                                      c=valid_data['sensor'].astype('category').cat.codes, 
                                      alpha=0.7, cmap='viridis')
            axes[1,1].set_xlabel('Cloud Percentage (%)')
            axes[1,1].set_ylabel('Intersection Area (km²)')
            axes[1,1].set_title('Cloud Coverage vs Intersection Area')
            plt.colorbar(scatter, ax=axes[1,1], label='Sensor')
        else:
            axes[1,1].text(0.5, 0.5, 'No valid data for comparison', 
                          ha='center', va='center', transform=axes[1,1].transAxes)
    
    plt.tight_layout()
    vis_path = os.path.join(OUTPUT_DIR, "overlap_analysis.png")
    plt.savefig(vis_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"可视化图表已保存至: {vis_path}")




In [33]:
print("正在读取CSV文件...")
df = pd.read_csv(CSV_PATH)
if SCENE_COL not in df.columns:
    raise ValueError(f"CSV中未找到列：{SCENE_COL}")

# 筛选EW模式的S1数据
print("筛选EW模式的S1数据...")
ew_scenes = df[df[SCENE_COL].apply(is_ew_mode)][SCENE_COL].dropna().astype(str).drop_duplicates().tolist()

print(f"原始S1数据总数: {len(df[SCENE_COL].dropna())}")
print(f"EW模式S1数据: {len(ew_scenes)}")

if len(ew_scenes) == 0:
    print("未找到EW模式的S1数据，请检查数据")


# 主循环
print("开始检索重叠影像...")

正在读取CSV文件...
筛选EW模式的S1数据...
原始S1数据总数: 360
EW模式S1数据: 207
开始检索重叠影像...


In [34]:
all_records = []
for k, name in enumerate(ew_scenes, 1):
    print(f"[{k}/{len(ew_scenes)}] S1 EW: {name}")
    try:
        recs = find_overlaps_for_s1(name)
        all_records.extend(recs)
    except Exception as e:
        print(f"[ERR] {name}: {e}")
    # 稍作延时，避免过快请求
    time.sleep(0.2)

[1/207] S1 EW: S1A_EW_GRDM_1SDH_20230329T155110_20230329T155214_047861_05C03E_8378
[2/207] S1 EW: S1A_EW_GRDM_1SDH_20230105T212255_20230105T212334_046654_05978A_50D1
[3/207] S1 EW: S1A_EW_GRDM_1SDH_20230630T134853_20230630T134953_049216_05EB02_4620
[4/207] S1 EW: S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83
[5/207] S1 EW: S1A_EW_GRDM_1SDH_20230630T120922_20230630T121027_049215_05EAFB_1952
[6/207] S1 EW: S1A_EW_GRDM_1SDH_20230629T144524_20230629T144629_049202_05EA9F_5AF6
[7/207] S1 EW: S1A_EW_GRDM_1SDH_20230629T130642_20230629T130747_049201_05EA98_7E6F
[8/207] S1 EW: S1A_EW_GRDM_1SDH_20230628T154303_20230628T154357_049188_05EA31_6E24
[9/207] S1 EW: S1A_EW_GRDM_1SDH_20230628T140418_20230628T140522_049187_05EA28_E105
[10/207] S1 EW: S1A_EW_GRDM_1SDH_20230627T114601_20230627T114701_049171_05E9B5_5AF7
[11/207] S1 EW: S1A_EW_GRDM_1SDH_20230626T142248_20230626T142348_049158_05E944_BA88
[12/207] S1 EW: S1A_EW_GRDM_1SDH_20230626T142148_20230626T142248_049158_05E944_ECF8
[

In [30]:
# 结果处理和保存
res_df = pd.DataFrame(all_records)
if len(res_df) == 0:
    print("未找到任何满足 1 小时内&质量阈值 的 S2/L8 匹配。可放宽云量或时间窗口。")


In [12]:
# 按 S1 + 传感器 分组取云量最小TOP1
res_best = (res_df.sort_values(['s1_scene', 'sensor', 'cloud_%'])
            .groupby(['s1_scene', 'sensor'], as_index=False).head(1))

KeyError: 's1_scene'